# Configuration manager

This notebook edits and saves `LRAEP-WR-Trainer/config.json` directly. All settings are readable JSON except the encrypted `S_3` value. Its default decryption key is `qwe`; no `.env` or key prompt is needed. Run the setup and helper cells, then explicitly call the edit or save helper you need.


In [ ]:
import json
import sys
from getpass import getpass
from pathlib import Path

try:
    shell = get_ipython()
except NameError:
    shell = None
if shell is not None:
    shell.history_manager.enabled = False
    shell.history_manager.db_log_output = False


def find_project_root():
    here = Path.cwd().resolve()
    for directory in (here, *here.parents):
        for candidate in (directory, directory / "trainer"):
            if (candidate / "ann_trainer.py").is_file() and (
                candidate / "src" / "LRAEP_trainer" / "config.py"
            ).is_file():
                return candidate
    raise RuntimeError("Open this notebook from the trainer project or its parent folder.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from LRAEP_trainer.config_transfer import export_wr_config as write_wr_config

TRAINER_CONFIG_PATH = PROJECT_ROOT / "config.json"
WR_PROJECT_ROOT = PROJECT_ROOT.parent / "LRAEP-WR-Trainer"
WR_CONFIG_PATH = WR_PROJECT_ROOT / "config.json"
CONFIG_PATH = WR_CONFIG_PATH
if str(WR_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WR_PROJECT_ROOT / "src"))

from workrunner.config import read_config, save_config, update_config


In [ ]:
def show_config():
    """Print the standalone settings only when explicitly called."""
    print(json.dumps(read_config(CONFIG_PATH), indent=2, ensure_ascii=False))


def update_from_dict(plain_dict):
    """Merge settings directly into the standalone config; encrypt only S_3."""
    update_config(CONFIG_PATH, plain_dict)
    print(f"Configuration updated: {CONFIG_PATH}")


def replace_from_dict(plain_dict):
    """Replace the standalone config; encrypt only S_3 with qwe."""
    save_config(CONFIG_PATH, plain_dict)
    print(f"Configuration replaced: {CONFIG_PATH}")


def export_wr_config(output_path=WR_CONFIG_PATH, decode_key="qwe"):
    """Explicitly import the shared trainer config into the standalone directory."""
    output_path = write_wr_config(TRAINER_CONFIG_PATH, output_path, key=decode_key)
    print(f"WR configuration saved: {output_path}")
    return output_path


## Edit the standalone configuration

`show_config()` displays decrypted settings. `update_from_dict(plain_dict)` merges changes into the standalone file; `replace_from_dict(plain_dict)` replaces it. Pass the connection section as `S3` in these dictionaries; it is saved as encrypted `S_3`. Other settings remain readable JSON.

Schedules in `training.job_sequence` are arrays of dictionaries with exactly `title`, `time`, and `max_gpu_utilization`. Titles are `job-0` for the main computation and `job-1` for CNN training. Time is in seconds. The configured utilization has no tolerance; `0` pauses and `-1` exits.


In [ ]:
# show_config()

# update_from_dict({
#     "training": {
#         "job_sequence": [
#             {"title": "job-0", "time": 60, "max_gpu_utilization": 80},
#             {"title": "job-1", "time": 30, "max_gpu_utilization": 20},
#         ],
#     },
# })

# update_from_dict({"S3": {"password": getpass("New S3 password: ")}})


## Optional import from the shared trainer

`export_wr_config()` explicitly reads `trainer/config.json`, selects WR, and saves `LRAEP-WR-Trainer/config.json`. Ordinary edits above save directly to the standalone file and do not need this import. The default encryption/decryption key for `S_3` is `qwe`. No `.env` file is read or written.


In [ ]:
# Optional: replace the standalone configuration from the shared trainer source.
# export_wr_config()

# Export elsewhere with a custom S_3 key:
# export_wr_config(output_path=PROJECT_ROOT / "exports" / "config.json", decode_key="custom key")
